In [11]:
# NNSE Function-based Implementation for Tyson Model
# Implements a vector-based mutation and permutation algorithm

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import copy

# ============================================================================
# === CONFIGURATION VARIABLES ===
# ============================================================================

# Simulation settings
N_STEPS = 100  # Number of mutation steps to run
SIGMA = 0.05  # Standard deviation for Gaussian mutations in normalized space
N_Vec = 50  # Number of bins for binning squared differences
MAX_VALUE = 2.0  # Maximum log value for bin thresholds (logspace goes from 0 to this)
T_START = 0.0  # Simulation start time
T_END = 500.0  # Simulation end time
N_TIME_POINTS = 501  # Number of time points

# Define bin thresholds using logspace (equally spaced in log space)
# y0, y1, ..., yN_BINS: thresholds for binning
bin_thresholds = np.linspace(0, MAX_VALUE, N_Vec + 1) + MAX_VALUE / N_Vec
  # y0, y1, ..., yN_BINS, basically sets y0 to the value of y1 because function will likely never be zero

# ============================================================================
# === BASE PARAMETERS (p0) ===
# ============================================================================

p0 = {
    "k1_aa_over_CT": 0.015,
    "k2": 0.0,
    "k3_CT": 200.0,
    "k4": 180.0,
    "k4prime": 0.018,
    "k5_minusP": 0.0,
    "k6": 1.0,
    "k7": 0.6,
    "k8_minusP": 100.0,
    "k9": 50.0,
    "CT": 1.0
}

# Parameters to vary
param_names = [
    "k1_aa_over_CT",
    "k3_CT",
    "k4",
    "k4prime",
    "k6",
    "k7"
]

p0_vec = np.array([p0[name] for name in param_names])
n_params = len(param_names)

# Time evaluation array
t_eval = np.linspace(T_START, T_END, N_TIME_POINTS)

# ============================================================================
# === TYSON MODEL DEFINITION ===
# ============================================================================

CT = p0["CT"]

def F_M(M, p):
    """Helper function for M-dependent rate"""
    return p["k4prime"] + p["k4"] * (M / p["CT"])**2

def f_rhs(t, x, p):
    """Right-hand side of the ODE system"""
    # x = [C2, CP, pM, M, Y, YP]
    C2, CP, pM, M, Y, YP = x
    k3 = p["k3_CT"] / p["CT"]
    k1 = p["k1_aa_over_CT"] * p['CT']
    dC2 = p["k6"] * M - p["k8_minusP"] * C2 + p["k9"] * CP
    dCP = -k3 * CP * Y + p["k8_minusP"] * C2 - p["k9"] * CP
    dpM = k3 * CP * Y - pM * F_M(M, p) + p["k5_minusP"] * M
    dM  = pM * F_M(M, p) - p["k5_minusP"] * M - p["k6"] * M
    dY  = k1 - p["k2"] * Y - k3 * CP * Y
    dYP = p["k6"] * M - p["k7"] * YP
    return np.array([dC2, dCP, dpM, dM, dY, dYP])

def simulate_at_params(p_local, t_eval):
    """Simulate the ODE system with given parameters"""
    y0 = np.array([0.9, 0.05, 0.0, 0.005, 0.3, 0.0])
    sol = solve_ivp(lambda tt, xx: f_rhs(tt, xx, p_local), 
                    (t_eval[0], t_eval[-1]), y0,
                    method='BDF', t_eval=t_eval, rtol=1e-6, atol=1e-8)
    if not sol.success:
        raise RuntimeError("Integrator failed: " + sol.message)
    return sol.t, sol.y

def compute_obs(X):
    """Compute observables: YT/CT and M/CT"""
    if X.ndim == 1:
        C2, CP, pM, M, Y, YP = X
        YT = Y + YP + pM + M
        return YT / CT, M / CT
    else:
        C2, CP, pM, M, Y, YP = X
        YT = Y + YP + pM + M
        return YT / CT, M / CT

# ============================================================================
# === REFERENCE SIMULATION (p0) ===
# ============================================================================

print("Running reference simulation with p0...")
t0, y0 = simulate_at_params(p0, t_eval)
YT0, M0 = compute_obs(y0)
print(f"✓ Reference simulation complete")

# ============================================================================
# === SIM FUNCTION ===
# ============================================================================

def sim(P_vec):
    """
    Simulate with parameter vector P and compute squared difference with p0.
    Returns the squared difference (f(xi) value).
    """
    # Convert parameter vector to dictionary
    p_local = copy.deepcopy(p0)
    for i, name in enumerate(param_names):
        p_local[name] = P_vec[i]
    
    # Run simulation
    try:
        t, y = simulate_at_params(p_local, t_eval)
        YT, M = compute_obs(y)
        
        # Interpolate reference to match time points
        YT0_interp = np.interp(t, t0, YT0)
        M0_interp = np.interp(t, t0, M0)
        
        # Compute squared differences
        diff_YT_sq = (YT - YT0_interp)**2
        diff_M_sq = (M - M0_interp)**2
        
        # Integrate squared differences
        integral_YT_sq = np.trapz(diff_YT_sq, t)
        integral_M_sq = np.trapz(diff_M_sq, t)
        squared_diff = integral_YT_sq + integral_M_sq
        
        return squared_diff
        
    except Exception as e:
        # If simulation fails, return infinity
        print(f"Warning: Simulation failed: {e}")
        return np.inf

print(f"✓ Configuration complete")
print(f"  Parameters: {param_names}")
print(f"  Number of parameters: {n_params}")
print(f"  Bin thresholds (y0, ..., y{N_Vec}): [{bin_thresholds[0]:.4e}, ..., {bin_thresholds[-1]:.4e}]")


Running reference simulation with p0...
✓ Reference simulation complete
✓ Configuration complete
  Parameters: ['k1_aa_over_CT', 'k3_CT', 'k4', 'k4prime', 'k6', 'k7']
  Number of parameters: 6
  Bin thresholds (y0, ..., y50): [4.0000e-02, ..., 2.0400e+00]


In [12]:
# ============================================================================
# === TYSONFUNC: MUTATION AND PERMUTATION FUNCTION ===
# ============================================================================

def TysonFunc(X_list, fX_list):
    """
    Mutate each parameter vector, evaluate, reject if worse, then permute.
    
    Args:
        X_list: List of parameter vectors [x0, x1, ..., xn] where each xi is a numpy array
        fX_list: List of function values [f(x0), f(x1), ..., f(xn)]
    
    Returns:
        v_list: List of parameter vectors after mutation and permutation [v0, v1, ..., vn]
        fv_list: List of function values [f(v0), f(v1), ..., f(vn)]
    """
    n = len(X_list)
    
    # Step 1: Mutate each xi
    X_prime_list = []
    fX_prime_list = []
    
    for i in range(n):
        xi = X_list[i]
        fxi = fX_list[i]
        
        # Normalize parameters: u_i = p_i / (2 * p0_i) so each lies in [0,1]
        u_vec = xi / (2.0 * p0_vec)
        
        # Apply Gaussian mutation in u-space
        u_mutated = u_vec + np.random.normal(0, SIGMA, size=n_params)
        
        # Wrap around boundaries [0, 1] with periodic boundary conditions
        u_mutated = u_mutated % 1.0
        
        # Map back to parameter space: p_i = 2 * p0_i * u_i
        xi_prime = 2.0 * p0_vec * u_mutated
        
        # Evaluate mutated parameter: f(x'i) := sim(x'i)
        fxi_prime = sim(xi_prime)
        
        # Step 2: Reject x'i if f(x'i) > f(xi), otherwise accept
        if fxi_prime > fxi:
            # Reject: keep original
            X_prime_list.append(xi.copy())
            fX_prime_list.append(fxi)
        else:
            # Accept: use mutated
            X_prime_list.append(xi_prime)
            fX_prime_list.append(fxi_prime)
    
    # Step 3: Permutation step
    # For each position i from n-1 down to 1 (0-indexed: n-1, n-2, ..., 1)
    # If f(x'_i) <= y_{i-1}, swap position i with position i-1
    v_list = X_prime_list.copy()
    fv_list = fX_prime_list.copy()
    
    # Track swaps
    swaps = []  # List of (i, j) tuples for swaps
    
    for i in range(n-1, 0, -1):  # i from n-1 down to 1
        if fv_list[i] <= bin_thresholds[i-1]:  # f(x'_i) <= y_{i-1}
            # Swap position i with position i-1
            v_list[i], v_list[i-1] = v_list[i-1].copy(), v_list[i].copy()
            fv_list[i], fv_list[i-1] = fv_list[i-1], fv_list[i]
            swaps.append((i, i-1))  # Record the swap
    
    return v_list, fv_list, swaps

print("✓ TysonFunc defined")


✓ TysonFunc defined


In [ ]:
# ============================================================================
# === SIMULATION LOOP ===
# ============================================================================

n = N_Vec  # Number of parameter vectors to maintain

print(f"Starting simulation with n={n} parameter vectors for {N_STEPS} steps...")

# Initialize: Generate n random parameter points in the unit cube and scale to actual cube
X_list = []
fX_list = []

print("Generating initial random parameter vectors...")
# Print table header
print(f"\n{'Index':<8} {'f(xi)':<15}")
print("-" * 23)

for i in range(n):
    # Generate random point in normalized u-space [0, 1]
    u_random = np.random.uniform(0, 1, size=n_params)
    # Map to parameter space: p_i = 2 * p0_i * u_i
    xi = 2.0 * p0_vec * u_random
    X_list.append(xi)
    
    # For first run, we need to compute f(xi) - accept all initially
    fxi = sim(xi)
    fX_list.append(fxi)
    
    # Print in table format (show all if n <= 20, otherwise show first 5, ..., last 5)
    if n <= 20:
        print(f"{i:<8} {fxi:<15.3e}")
    else:
        if i < 5 or i >= n - 5:
            print(f"{i:<8} {fxi:<15.3e}")
        elif i == 5:
            print(f"{'...':<8} {'...':<15}")

print(f"\n✓ Initialization complete")

# Sort by f(xi) from lowest to highest, keeping xi and f(xi) paired
# Create list of tuples (fxi, xi), sort by fxi, then unpack
paired = list(zip(fX_list, X_list))
paired_sorted = sorted(paired, key=lambda x: x[0])  # Sort by f(xi)
fX_list, X_list = zip(*paired_sorted)
fX_list = list(fX_list)
X_list = [x.copy() for x in X_list]  # Make sure we have copies

# Storage for tracking
all_X = [X_list.copy()]
all_fX = [fX_list.copy()]
all_swaps = [[]]  # Store swaps for each step (empty for initial state)

# Progress tracking
print_interval = max(1, N_STEPS // 20)  # Print every 5%

# Create table header
# Show first 5, middle, and last f values, or all if n is small
if n <= 7:
    # Show all f values
    header_cols = [f"f(x{i})" for i in range(n)]
    header = f"{'Step':<6} " + " ".join([f"{col:<12}" for col in header_cols]) + f" {'#swap':<6} {'swaps':<20}"
    separator_len = 6 + 1 + n * 13 + 1 + 6 + 1 + 20  # Step + space + n columns + #swap + swaps
else:
    # Show first 5, middle, last
    header_cols = [f"f(x{i})" for i in range(5)] + [f"f(x{n//2})", f"f(x{n-1})"]
    header = f"{'Step':<6} " + " ".join([f"{col:<12}" for col in header_cols]) + f" {'#swap':<6} {'swaps':<20}"
    separator_len = 6 + 1 + 7 * 13 + 1 + 6 + 1 + 20  # Step + space + 7 columns + #swap + swaps
print(f"\n{header}")
print("-" * separator_len)

# Print initial state
if n <= 7:
    f_vals_str = " ".join([f"{fx:<12.3e}" for fx in fX_list])
    print(f"{'Init':<6} {f_vals_str} {'0':<6} {'':<20}")
else:
    # Show first 5, middle, last
    f_vals = [fX_list[i] for i in range(5)] + [fX_list[n//2], fX_list[n-1]]
    f_vals_str = " ".join([f"{fx:<12.3e}" for fx in f_vals])
    print(f"{'Init':<6} {f_vals_str} {'0':<6} {'':<20}")

# Main simulation loop
for step in range(N_STEPS):
    # Apply TysonFunc
    X_list, fX_list, swaps = TysonFunc(X_list, fX_list)
    
    # Store trajectory
    all_X.append([x.copy() for x in X_list])
    all_fX.append(fX_list.copy())
    all_swaps.append(swaps)
    
    # Progress report in table format
    if (step + 1) % print_interval == 0 or step == 0:
        # Format swaps as "i-j, k-l, ..."
        swaps_str = ", ".join([f"{i}-{j}" for i, j in swaps]) if swaps else ""
        if len(swaps_str) > 20:
            swaps_str = swaps_str[:17] + "..."
        
        if n <= 7:
            f_vals_str = " ".join([f"{fx:<12.3e}" for fx in fX_list])
            print(f"{step+1:<6} {f_vals_str} {len(swaps):<6} {swaps_str:<20}")
        else:
            # Show first 5, middle, last
            f_vals = [fX_list[i] for i in range(5)] + [fX_list[n//2], fX_list[n-1]]
            f_vals_str = " ".join([f"{fx:<12.3e}" for fx in f_vals])
            print(f"{step+1:<6} {f_vals_str} {len(swaps):<6} {swaps_str:<20}")

print(f"\n✓ Simulation complete!")
# Print final state in table format
if n <= 7:
    f_vals_str = " ".join([f"{fx:<12.3e}" for fx in fX_list])
    print(f"{'Final':<6} {f_vals_str} {'0':<6} {'':<20}")
else:
    # Show first 5, middle, last
    f_vals = [fX_list[i] for i in range(5)] + [fX_list[n//2], fX_list[n-1]]
    f_vals_str = " ".join([f"{fx:<12.3e}" for fx in f_vals])
    print(f"{'Final':<6} {f_vals_str} {'0':<6} {'':<20}")

# Convert to numpy arrays for easier analysis
all_X_array = np.array(all_X)  # Shape: (N_STEPS+1, n, n_params)
all_fX_array = np.array(all_fX)  # Shape: (N_STEPS+1, n)


Starting simulation with n=50 parameter vectors for 100 steps...
Generating initial random parameter vectors...

Index    f(xi)          
-----------------------
0        2.126e+00      
1        6.840e+03      
2        7.851e+00      
3        2.804e+02      
4        5.723e+00      
...      ...            
45       1.064e+02      
46       6.325e+00      
47       6.925e+00      
48       1.766e+01      
49       1.108e+01      

✓ Initialization complete

Step   f(x0)        f(x1)        f(x2)        f(x3)        f(x4)        f(x25)       f(x49)       #swap  swaps               
------------------------------------------------------------------------------------------------------------------------------
Init   2.126e+00    4.260e+00    4.511e+00    4.914e+00    5.723e+00    1.544e+01    6.840e+03    0                          


In [ ]:
# ============================================================================
# === VISUALIZATION: PCA TRAJECTORY AND DISTRIBUTIONS ===
# ============================================================================

from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde

print("Preparing data for visualization...")

# Extract the best vector (lowest f value) at each step
# Since vectors are sorted by f value, index 0 is always the best
# all_X_array shape: (N_STEPS+1, n, n_params) -> extract best: (N_STEPS+1, n_params)
trajectory = all_X_array[:, 0, :]  # Best parameter vector at each step
squared_diffs = all_fX_array[:, 0]  # Best f value at each step

print(f"  Total steps: {len(trajectory)}")
print(f"  Best f value trajectory: {len(squared_diffs)} points")

# ============================================================================
# === PCA TRAJECTORY PLOT ===
# ============================================================================

print("\nComputing PCA for trajectory visualization...")

# Normalize trajectory for PCA (use standardized parameters)
trajectory_normalized = (trajectory - trajectory.mean(axis=0)) / (trajectory.std(axis=0) + 1e-10)

# Compute PCA
pca = PCA(n_components=min(3, n_params))
trajectory_pca = pca.fit_transform(trajectory_normalized)

print(f"✓ PCA complete")
print(f"  Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"  Total explained variance: {np.sum(pca.explained_variance_ratio_):.4f}")

# Simulate with first and last parameters for YT/CT comparison
print("Simulating with first and last parameters...")
P_first = trajectory[0]  # Best parameter vector at step 0
P_last = trajectory[-1]  # Best parameter vector at final step

# Convert to parameter dictionaries
p_first = copy.deepcopy(p0)
p_last = copy.deepcopy(p0)
for i, name in enumerate(param_names):
    p_first[name] = P_first[i]
    p_last[name] = P_last[i]

# Simulate
t_first, y_first = simulate_at_params(p_first, t_eval)
YT_first, M_first = compute_obs(y_first)
t_last, y_last = simulate_at_params(p_last, t_eval)
YT_last, M_last = compute_obs(y_last)

print("✓ Simulations complete")

# Create figure with 3 subplots
fig = plt.figure(figsize=(20, 6))
gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

# Plot 2D PCA trajectory
ax1 = fig.add_subplot(gs[0, 0])
scatter = ax1.scatter(trajectory_pca[:, 0], trajectory_pca[:, 1], 
                     c=range(len(trajectory_pca)), cmap='viridis', 
                     s=20, alpha=0.6, edgecolors='none')
ax1.plot(trajectory_pca[:, 0], trajectory_pca[:, 1], 'k-', alpha=0.3, linewidth=0.5)
ax1.scatter(trajectory_pca[0, 0], trajectory_pca[0, 1], 
           color='red', s=100, marker='o', label='Start', zorder=5)
ax1.scatter(trajectory_pca[-1, 0], trajectory_pca[-1, 1], 
           color='blue', s=100, marker='s', label='End', zorder=5)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
ax1.set_title('NNSE Trajectory in PCA Space (PC1 vs PC2)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax1, label='Step')

# Second subplot: 3D plot or squared difference over time
if trajectory_pca.shape[1] >= 3:
    from mpl_toolkits.mplot3d import Axes3D
    ax2 = fig.add_subplot(gs[0, 1], projection='3d')
    scatter2 = ax2.scatter(trajectory_pca[:, 0], trajectory_pca[:, 1], trajectory_pca[:, 2],
                         c=range(len(trajectory_pca)), cmap='viridis', s=20, alpha=0.6)
    ax2.plot(trajectory_pca[:, 0], trajectory_pca[:, 1], trajectory_pca[:, 2], 
            'k-', alpha=0.3, linewidth=0.5)
    ax2.scatter(trajectory_pca[0, 0], trajectory_pca[0, 1], trajectory_pca[0, 2],
               color='red', s=100, marker='o', label='Start')
    ax2.scatter(trajectory_pca[-1, 0], trajectory_pca[-1, 1], trajectory_pca[-1, 2],
               color='blue', s=100, marker='s', label='End')
    ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})', fontsize=10)
    ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})', fontsize=10)
    ax2.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.2%})', fontsize=10)
    ax2.set_title('NNSE Trajectory in PCA Space (3D)', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    plt.colorbar(scatter2, ax=ax2, label='Step')
else:
    # If only 2 components, show squared difference over time
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(range(len(squared_diffs)), squared_diffs, 'b-', linewidth=1, alpha=0.7)
    ax2.set_xlabel('Step', fontsize=12)
    ax2.set_ylabel('Squared Difference', fontsize=12)
    ax2.set_title('Best Squared Difference Over Time', fontsize=14, fontweight='bold')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)

# Third subplot: YT/CT comparison
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(t0, YT0, 'k-', lw=2, label='Reference (p0)', alpha=0.8)
ax3.plot(t_first, YT_first, 'r-', lw=2, label='First parameter', alpha=0.7)
ax3.plot(t_last, YT_last, 'b-', lw=2, label='Last parameter', alpha=0.7)
ax3.set_xlabel('Time (min)', fontsize=12)
ax3.set_ylabel('YT/CT', fontsize=12)
ax3.set_title('YT/CT Comparison', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# === DISTRIBUTION OF SQUARED DIFFERENCES ===
# ============================================================================

print("\nPlotting distribution of squared differences...")

# For distribution, use all squared differences from all vectors at all steps
# This gives a better picture of the full distribution
squared_diffs_all = all_fX_array.flatten()  # All function values from all vectors

# Filter out infinite values
valid_mask = np.isfinite(squared_diffs_all)
squared_diffs_valid = squared_diffs_all[valid_mask]

print(f"  Valid values: {np.sum(valid_mask)}/{len(squared_diffs_all)}")
print(f"  Mean: {np.mean(squared_diffs_valid):.6e}")
print(f"  Median: {np.median(squared_diffs_valid):.6e}")
print(f"  Min: {np.min(squared_diffs_valid):.6e}")
print(f"  Max: {np.max(squared_diffs_valid):.6e}")

# Create distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram with linear bins
ax1 = axes[0]
n_bins_hist = 50
# Use linear bins from min to max
counts, bins_hist, patches = ax1.hist(squared_diffs_valid, bins=n_bins_hist, 
                                      edgecolor='black', alpha=0.7, color='steelblue',
                                      density=True)
ax1.axvline(np.mean(squared_diffs_valid), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {np.mean(squared_diffs_valid):.4e}')
ax1.axvline(np.median(squared_diffs_valid), color='green', linestyle='--', 
           linewidth=2, label=f'Median: {np.median(squared_diffs_valid):.4e}')
ax1.set_xlabel('Squared Difference', fontsize=12, fontweight='bold')
ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of Squared Differences (Histogram)', 
             fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Kernel density estimate (KDE) with linear scale
ax2 = axes[1]
if len(squared_diffs_valid) > 1:
    # Use linear KDE (not log-transformed)
    kde = gaussian_kde(squared_diffs_valid)
    x_kde = np.linspace(squared_diffs_valid.min(), squared_diffs_valid.max(), 200)
    density = kde(x_kde)
    ax2.plot(x_kde, density, 'b-', linewidth=2, label='KDE')
    ax2.fill_between(x_kde, 0, density, alpha=0.3, color='steelblue')
    ax2.axvline(np.mean(squared_diffs_valid), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(squared_diffs_valid):.4e}')
    ax2.axvline(np.median(squared_diffs_valid), color='green', linestyle='--', 
               linewidth=2, label=f'Median: {np.median(squared_diffs_valid):.4e}')
    ax2.set_xlabel('Squared Difference', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of Squared Differences (KDE)', 
                 fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Not enough data for KDE', 
            ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title('Distribution of Squared Differences (KDE)', 
                 fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")
